In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import json
import os

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

from tensorflow.keras.optimizers import Adam, SGD, RMSprop, Adagrad

In [2]:
import sys
sys.path.append("../training")

from data_generator import DataGenerator

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

DATA_DIR = "../data"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
CSV_PATH = os.path.join(DATA_DIR, "train_solution_bounding_boxes.csv")

df = pd.read_csv(CSV_PATH)
df["image_path"] = df["image"].apply(lambda x: os.path.join(TRAIN_DIR, x))

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

train_df.reset_index(drop=True, inplace=True)
val_df.reset_index(drop=True, inplace=True)

In [4]:
BATCH_SIZE = 16

train_generator = DataGenerator(train_df, BATCH_SIZE, augment=True)
val_generator = DataGenerator(val_df, BATCH_SIZE, augment=False)

In [5]:
def create_model():
    
    base_model = ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=(224,224,3)
    )
    
    base_model.trainable = False
    
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.3)(x)
    
    output = Dense(4, activation='sigmoid')(x)
    
    model = Model(inputs=base_model.input, outputs=output)
    
    return model

In [6]:
LOG_DIR = "../logs"
os.makedirs(LOG_DIR, exist_ok=True)

In [7]:
def train_optimizer(optimizer, name):
    
    model = create_model()
    
    model.compile(
        optimizer=optimizer,
        loss='mse',
        metrics=['mae']
    )
    
    history = model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=5
    )
    
    model.save(f"../models/resnet50_{name}.h5")
    
    with open(f"../logs/{name}_history.json", "w") as f:
        json.dump(history.history, f)
    
    return history

In [8]:
history_adam = train_optimizer(
    Adam(learning_rate=0.0001),
    "adam"
)

c:\Users\Admin\anaconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5
27/27 ━━━━━━━━━━━━━━━━━━━━ 68s 2s/step - loss: 0.0692 - mae: 0.2014 - val_loss: 0.0445 - val_mae: 0.1394
Epoch 2/5
27/27 ━━━━━━━━━━━━━━━━━━━━ 45s 2s/step - loss: 0.0633 - mae: 0.1899 - val_loss: 0.0474 - val_mae: 0.1466
Epoch 3/5
27/27 ━━━━━━━━━━━━━━━━━━━━ 45s 2s/step - loss: 0.0593 - mae: 0.1846 - val_loss: 0.0508 - val_mae: 0.1502
Epoch 4/5
27/27 ━━━━━━━━━━━━━━━━━━━━ 44s 2s/step - loss: 0.0603 - mae: 0.1806 - val_loss: 0.0440 - val_mae: 0.1406
Epoch 5/5
27/27 ━━━━━━━━━━━━━━━━━━━━ 43s 2s/step - loss: 0.0574 - mae: 0.1771 - val_loss: 0.0431 - val_mae: 0.1379


In [9]:
history_sgd = train_optimizer(
    SGD(learning_rate=0.001),
    "sgd"
)

Epoch 1/5
27/27 ━━━━━━━━━━━━━━━━━━━━ 71s 2s/step - loss: 0.0824 - mae: 0.2362 - val_loss: 0.0523 - val_mae: 0.1829
Epoch 2/5
27/27 ━━━━━━━━━━━━━━━━━━━━ 45s 2s/step - loss: 0.0697 - mae: 0.2089 - val_loss: 0.0456 - val_mae: 0.1506
Epoch 3/5
27/27 ━━━━━━━━━━━━━━━━━━━━ 42s 2s/step - loss: 0.0655 - mae: 0.1979 - val_loss: 0.0440 - val_mae: 0.1392
Epoch 4/5
27/27 ━━━━━━━━━━━━━━━━━━━━ 42s 2s/step - loss: 0.0675 - mae: 0.2004 - val_loss: 0.0434 - val_mae: 0.1381
Epoch 5/5
27/27 ━━━━━━━━━━━━━━━━━━━━ 43s 2s/step - loss: 0.0662 - mae: 0.2001 - val_loss: 0.0434 - val_mae: 0.1384


In [10]:
history_rms = train_optimizer(
    RMSprop(learning_rate=0.0001),
    "rmsprop"
)

Epoch 1/5
27/27 ━━━━━━━━━━━━━━━━━━━━ 77s 2s/step - loss: 0.0697 - mae: 0.2029 - val_loss: 0.0438 - val_mae: 0.1423
Epoch 2/5
27/27 ━━━━━━━━━━━━━━━━━━━━ 45s 2s/step - loss: 0.0630 - mae: 0.1925 - val_loss: 0.0435 - val_mae: 0.1404
Epoch 3/5
27/27 ━━━━━━━━━━━━━━━━━━━━ 44s 2s/step - loss: 0.0635 - mae: 0.1926 - val_loss: 0.0433 - val_mae: 0.1390
Epoch 4/5
27/27 ━━━━━━━━━━━━━━━━━━━━ 46s 2s/step - loss: 0.0590 - mae: 0.1823 - val_loss: 0.0472 - val_mae: 0.1516
Epoch 5/5
27/27 ━━━━━━━━━━━━━━━━━━━━ 46s 2s/step - loss: 0.0597 - mae: 0.1831 - val_loss: 0.0446 - val_mae: 0.1407


In [11]:
history_ada = train_optimizer(
    Adagrad(learning_rate=0.01),
    "adagrad"
)

Epoch 1/5
27/27 ━━━━━━━━━━━━━━━━━━━━ 72s 2s/step - loss: 0.0709 - mae: 0.2069 - val_loss: 0.0496 - val_mae: 0.1469
Epoch 2/5
27/27 ━━━━━━━━━━━━━━━━━━━━ 43s 2s/step - loss: 0.0631 - mae: 0.1916 - val_loss: 0.0433 - val_mae: 0.1421
Epoch 3/5
27/27 ━━━━━━━━━━━━━━━━━━━━ 44s 2s/step - loss: 0.0648 - mae: 0.1917 - val_loss: 0.0433 - val_mae: 0.1400
Epoch 4/5
27/27 ━━━━━━━━━━━━━━━━━━━━ 47s 2s/step - loss: 0.0605 - mae: 0.1852 - val_loss: 0.0458 - val_mae: 0.1453
Epoch 5/5
27/27 ━━━━━━━━━━━━━━━━━━━━ 44s 2s/step - loss: 0.0578 - mae: 0.1766 - val_loss: 0.0468 - val_mae: 0.1421
